# Production Prompt Lifecycle

This notebook teaches you how to manage the **full prompt lifecycle** in production using [Langfuse](https://langfuse.com/) and AWS services. You will learn to version prompts, run systematic evaluations, automate quality scoring, and integrate everything into a CI/CD pipeline.

## Learning Objectives

By the end of this notebook, you will be able to:
- **Version and manage prompts** in Langfuse (not just Git)
- **Create systematic evaluation datasets** for repeatable testing
- **Implement automated quality scoring** with LLM-as-Judge
- **Integrate Langfuse with CI/CD pipelines** using AWS CodePipeline

## Why This Matters

Git tracks code changes, but it cannot tell you which prompt version produced which quality metrics in production. Langfuse bridges this gap by linking prompt versions to traces, costs, and latency data -- giving you a complete picture of how each prompt performs.

| Challenge | Git Alone | Git + Langfuse |
|-----------|-----------|----------------|
| Version tracking | File diffs | Prompt versions linked to traces |
| Performance analysis | Manual log review | Cost, latency, token usage per version |
| A/B testing | Feature flags | Label-based routing with analytics |
| Quality gates | Unit tests only | LLM-as-Judge + dataset experiments |

**Duration**: ~75-90 minutes

## Prerequisites

Before running this notebook, ensure you have:
1. Completed the [02-developer-journey](../02-developer-journey/) notebooks (Langfuse already configured)
2. A self-hosted or cloud Langfuse instance
3. AWS credentials with Bedrock access

## Setup

In [ ]:
from __future__ import annotations

from dotenv import load_dotenv

load_dotenv()

import json
import os
import time
from pathlib import Path

import boto3

# ── AWS Bedrock client ──────────────────────────────────────────────────
REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

# Model configuration
MODEL_SONNET = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"
MODEL_HAIKU = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_ID = MODEL_SONNET        # Primary model for agent responses
JUDGE_MODEL_ID = MODEL_HAIKU   # Cheaper model for LLM-as-Judge

# ── Langfuse client ──────────────────────────────────────────────────
from langfuse import Langfuse

langfuse = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host=os.environ.get("LANGFUSE_BASE_URL", "https://cloud.langfuse.com"),
)

# ── Helper: call Bedrock Converse API ─────────────────────────────────


def call_bedrock(
    prompt: str,
    system: str | None = None,
    temperature: float = 0.0,
    max_tokens: int = 500,
    model_id: str | None = None,
) -> dict:
    """Call Bedrock Converse API and return text + usage metrics."""
    model = model_id or MODEL_ID
    messages = [{"role": "user", "content": [{"text": prompt}]}]

    kwargs: dict = {
        "modelId": model,
        "messages": messages,
        "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature},
    }
    if system:
        kwargs["system"] = [{"text": system}]

    response = bedrock_runtime.converse(**kwargs)
    usage = response["usage"]

    return {
        "text": response["output"]["message"]["content"][0]["text"],
        "input_tokens": usage["inputTokens"],
        "output_tokens": usage["outputTokens"],
        "latency_ms": response["metrics"]["latencyMs"],
    }


print(f"Region:       {REGION}")
print(f"Agent Model:  {MODEL_ID}")
print(f"Judge Model:  {JUDGE_MODEL_ID}")
print(f"Langfuse:     {langfuse.base_url}")
print("\nSetup complete!")

---

# Section 1: Langfuse Prompt Management (20 min)

Langfuse Prompt Management lets you store prompt versions alongside their performance data. Unlike Git, which only tracks text changes, Langfuse links each prompt version to the traces it produced -- giving you cost, latency, and quality metrics per version.

| Concept | Description |
|---------|-------------|
| **Why Langfuse vs Git** | Git versions code; Langfuse links prompts to traces for performance analysis |
| **Production labels** | Mark versions as `production` for safe, label-based deployments |
| **Prompt-trace linkage** | Every trace records which prompt version was used |

**References:**
- [Langfuse Prompt Management](https://langfuse.com/docs/prompts/get-started)
- [Linking Prompts to Traces](https://langfuse.com/docs/prompts/get-started#link-prompts-to-traces)

## Demo 1: Create Prompt Versions in Langfuse

We will create two versions of a customer support system prompt:

- **v1 (production)**: Verbose, detailed instructions — currently serving traffic
- **v2 (draft)**: Concise candidate — we want to evaluate before promoting

Calling `create_prompt` with the same `name` automatically creates a new version.

In [ ]:
# ── Prompt definitions ─────────────────────────────────────────────────

PROMPT_NAME = "customer-support-system"

PROMPT_V1_TEXT = """You are a helpful customer support assistant for TechMart Electronics.

Your role is to assist customers with:
- Product information and specifications
- Return policies and procedures
- Technical troubleshooting
- Order status inquiries

Always be polite, professional, and thorough in your responses.
If you don't know the answer, say so and offer to escalate to a human agent.
Provide detailed explanations and walk the customer through each step."""

PROMPT_V2_TEXT = """You are a customer support assistant for TechMart Electronics.
Help with: product info, returns, tech support, order status.
Be professional and concise. If unsure, offer to escalate to a human agent."""

# Model configuration stored alongside the prompt
CONFIG_V1 = {
    "model": "global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "temperature": 0.0,
    "max_tokens": 1000,
}

CONFIG_V2 = {
    "model": "global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "temperature": 0.0,
    "max_tokens": 1000,
}

# ── Create prompt versions ─────────────────────────────────────────────

# v1: Verbose prompt currently serving production traffic
langfuse.create_prompt(
    name=PROMPT_NAME,
    type="text",
    prompt=PROMPT_V1_TEXT,
    config=CONFIG_V1,
    labels=["production"],
)
print("Created prompt v1 (production)")

# v2: Concise candidate we want to evaluate before promoting
langfuse.create_prompt(
    name=PROMPT_NAME,
    type="text",
    prompt=PROMPT_V2_TEXT,
    config=CONFIG_V2,
    labels=["draft"],
)
print("Created prompt v2 (draft)")

## Demo 2: Fetch and Test Prompt Versions

In a production workflow, prompt versions are labeled (e.g., `draft`, `production`) and fetched by label at runtime. This lets you evaluate a draft candidate against the current production version before promoting it -- without changing any application code.

In [ ]:
# ── Fetch prompt versions by label ─────────────────────────────────────

prompt_prod = langfuse.get_prompt(PROMPT_NAME, label="production")
prompt_draft = langfuse.get_prompt(PROMPT_NAME, label="draft")

system_prompt_prod = prompt_prod.compile()
system_prompt_draft = prompt_draft.compile()
config_prod = prompt_prod.config
config_draft = prompt_draft.config

print(f"Production prompt (v{prompt_prod.version}):")
print(f"  Labels: {prompt_prod.labels}")
print(f"  Config: {json.dumps(config_prod, indent=2)}")
print(f"  Text:   {system_prompt_prod[:80]}...")

print(f"\nDraft candidate (v{prompt_draft.version}):")
print(f"  Labels: {prompt_draft.labels}")
print(f"  Config: {json.dumps(config_draft, indent=2)}")
print(f"  Text:   {system_prompt_draft[:80]}...")

# ── Test the draft candidate ────────────────────────────────────────────

user_message = "What is your return policy for laptops?"

with langfuse.start_as_current_observation(
    as_type="span",
    name="draft-candidate-test",
) as trace:
    with langfuse.start_as_current_observation(
        as_type="generation",
        name="llm-call",
        prompt=prompt_draft,
        input=user_message,
    ) as generation:
        response = call_bedrock(
            prompt=user_message,
            system=system_prompt_draft,
            temperature=config_draft.get("temperature", 0.0),
            max_tokens=config_draft.get("max_tokens", 500),
        )
        generation.update(output=response["text"])
    trace.update(output=response["text"])
langfuse.flush()

print(f"\nQuery: {user_message}")
print(f"Response: {response['text'][:300]}...")
print(f"\nTokens: input={response['input_tokens']}, output={response['output_tokens']}")
print(f"Latency: {response['latency_ms']}ms")

## Demo 3: Analyze Performance by Prompt Version

Let us run both prompt versions against the same queries and compare their token usage and latency. In production, you would view this comparison in the Langfuse dashboard.

In [ ]:
# ── Compare draft vs production performance ───────────────────────────

test_queries = [
    "What is your return policy for laptops?",
    "My laptop won't turn on, can you help?",
    "What smartphones do you carry?",
]

print("=" * 70)
print("PROMPT VERSION COMPARISON: draft (concise) vs production (verbose)")
print("=" * 70)

# Use the fetched prompts from Demo 2 (falls back to raw text if Langfuse unavailable)
versions = {
    "draft": {"prompt": system_prompt_draft, "config": config_draft},
    "production": {"prompt": system_prompt_prod, "config": config_prod},
}

results_by_version: dict[str, list[dict]] = {}

for version_name, version_data in versions.items():
    print(f"\n--- {version_name} ---")
    version_results = []

    for query in test_queries:
        resp = call_bedrock(
            prompt=query,
            system=version_data["prompt"],
            temperature=version_data["config"]["temperature"],
            max_tokens=version_data["config"]["max_tokens"],
        )
        version_results.append(resp)
        print(
            f"  [{resp['input_tokens']:>4} in, {resp['output_tokens']:>4} out, "
            f"{resp['latency_ms']:>5}ms] {query[:50]}"
        )

    results_by_version[version_name] = version_results

# ── Summary comparison ────────────────────────────────────────────────
print(f"\n{'=' * 70}")
print(f"{'Version':<15} {'Avg Input':>12} {'Avg Output':>12} {'Avg Latency':>14}")
print("-" * 70)

for version_name, results_list in results_by_version.items():
    avg_in = sum(r["input_tokens"] for r in results_list) / len(results_list)
    avg_out = sum(r["output_tokens"] for r in results_list) / len(results_list)
    avg_lat = sum(r["latency_ms"] for r in results_list) / len(results_list)
    print(f"{version_name:<15} {avg_in:>10.0f} tk {avg_out:>10.0f} tk {avg_lat:>11.0f} ms")

print("=" * 70)
print("\nThe draft (concise) candidate uses fewer system prompt tokens per request,")
print("potentially reducing costs. But does quality hold up? That is what Section")
print("2 will verify using systematic evaluation and automated scoring.")

---

# Section 2: Systematic Evaluation and Quality Scoring (35 min)

This section covers the full evaluation loop: create a dataset, run experiments, score results, and gate promotion decisions.

You will:
1. **Create an evaluation dataset** in Langfuse from local test cases
2. **Run the production baseline** experiment and review raw results
3. **Score with keyword matching** -- the simplest quality check
4. **Score with LLM-as-Judge** -- using Claude Haiku for nuanced assessment
5. **Run the draft candidate** with automated evaluators built-in
6. **Compare and gate** -- decide whether the draft is safe to promote

| Concept | Description |
|---------|-------------|
| **Datasets** | Test cases stored in Langfuse, linked to experiments for side-by-side comparison |
| **Keyword scoring** | Fast, deterministic check for expected content |
| **LLM-as-Judge** | Cheaper model (Haiku) assesses quality on a numeric scale |
| **Evaluators** | Scoring functions that run automatically inside `dataset.run_experiment()` |
| **Quality gate** | Draft must meet threshold AND not regress from production |

**References:**
- [Langfuse Datasets](https://langfuse.com/docs/datasets/overview)
- [Langfuse Scores](https://langfuse.com/docs/scores/overview)
- [LLM-as-Judge](https://langfuse.com/docs/scores/model-based-evals)

## Demo 1: Create Evaluation Dataset in Langfuse

We will load test cases from a local JSON file and upload them to Langfuse as a dataset. Each item has:
- **input**: The customer query
- **expected_output**: Keywords the response should contain, expected tool usage
- **metadata**: Category, complexity level

In [ ]:
# ── Load local evaluation dataset ─────────────────────────────────────

EVAL_DATASET_PATH = Path("data/customer-support-eval.json")
DATASET_NAME = "customer-support-eval"

with open(EVAL_DATASET_PATH, encoding="utf-8") as f:
    eval_test_cases = json.load(f)

print(f"Loaded {len(eval_test_cases)} test cases from {EVAL_DATASET_PATH}")
print("\nSample entries:")
for case in eval_test_cases[:3]:
    print(f"  [{case['metadata']['category']:>14}] {case['input']['query'][:55]}...")

# ── Upload to Langfuse as a dataset ────────────────────────────────────

langfuse.create_dataset(
    name=DATASET_NAME,
    description="Core evaluation scenarios for customer support agent",
    metadata={"version": "1.0", "created_by": "workshop"},
)

for case in eval_test_cases:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input=case["input"],
        expected_output=case.get("expected_output", {}),
        metadata=case.get("metadata", {}),
    )

langfuse.flush()
print(f"\nUploaded {len(eval_test_cases)} items to Langfuse dataset: {DATASET_NAME}")

## Demo 2: Run Production Baseline Experiment

We run the **current production prompt** against every dataset item and record the results. This gives us a baseline to compare against when we evaluate the draft candidate in Section 3.

In Langfuse, this creates a **dataset run** called `production-baseline` that links each trace back to the dataset item.

In [ ]:
# ── Run production baseline experiment (no scoring yet) ───────────────

langfuse_dataset = langfuse.get_dataset(DATASET_NAME)

production_results: list[dict] = []


def production_task(*, item, **kwargs):
    """Task function: call Bedrock with the production prompt."""
    query = item.input["query"]
    resp = call_bedrock(
        prompt=query,
        system=system_prompt_prod,
        max_tokens=config_prod.get("max_tokens", 1000),
    )

    # Report model usage to Langfuse for cost tracking
    with langfuse.start_as_current_observation(
        as_type="generation",
        name="bedrock-converse",
        model=MODEL_ID,
        input=query,
    ) as gen:
        gen.update(
            output=resp["text"],
            usage={"input": resp["input_tokens"], "output": resp["output_tokens"]},
        )

    production_results.append({
        "query": query,
        "response": resp["text"],
        "input_tokens": resp["input_tokens"],
        "output_tokens": resp["output_tokens"],
        "latency_ms": resp["latency_ms"],
        "expected": item.expected_output or {},
    })
    return resp["text"]


print("Running experiment: production-baseline...")
prod_experiment = langfuse_dataset.run_experiment(
    name="production-baseline",
    task=production_task,
    max_concurrency=1,
)
langfuse.flush()
print(f"  Completed {len(production_results)} items")

# Retrieve trace IDs from the dataset run for manual scoring later
prod_trace_ids: list[str] = []
try:
    time.sleep(2)  # Allow Langfuse to process
    run_items = langfuse.api.dataset_run_items.list(
        dataset_id=langfuse_dataset.id,
        run_name=prod_experiment.run_name,
    )
    prod_trace_ids = [item.trace_id for item in run_items.data]
    print(f"  Retrieved {len(prod_trace_ids)} trace IDs from run: {prod_experiment.run_name}")
except Exception as e:
    print(f"  [WARN] Could not retrieve trace IDs: {e}")

# Quick token summary
avg_in = sum(r["input_tokens"] for r in production_results) / len(production_results)
avg_out = sum(r["output_tokens"] for r in production_results) / len(production_results)
avg_lat = sum(r["latency_ms"] for r in production_results) / len(production_results)
print(f"  Avg tokens: {avg_in:.0f} in, {avg_out:.0f} out | Avg latency: {avg_lat:.0f}ms")

## Demo 3: Review Baseline Results

Before moving to scoring, let's inspect a few production responses to see what we're working with.

In [ ]:
# ── Preview production baseline responses ─────────────────────────────

from IPython.display import display, Markdown

md_parts = ["# Production Baseline: Sample Responses\n"]

for i, r in enumerate(production_results[:3]):
    expected_kw = r["expected"].get("should_contain", [])
    kw_line = f"**Expected keywords:** {expected_kw}" if expected_kw else ""

    md_parts.append(f"""---
### Item {i + 1}
**Query:** {r['query']}

**Tokens:** {r['input_tokens']} in, {r['output_tokens']} out | {r['latency_ms']}ms

{kw_line}

**Response:**

{r['response']}
""")

md_parts.append(f"---\n*... ({len(production_results)} items total)*\n\n**Next:** Demo 3 introduces scoring to measure quality systematically.")
display(Markdown("\n".join(md_parts)))

## Demo 3: Keyword-Based Scoring

The simplest scoring approach: check whether the response contains expected keywords from the evaluation dataset. We score the **production baseline** results and push scores to Langfuse manually.

In [ ]:
# ── Keyword-based scoring ─────────────────────────────────────────────


def score_keyword_accuracy(response_text: str, expected: dict) -> float:
    """Score based on whether expected keywords appear in the response.

    Returns a score between 0.0 and 1.0.
    """
    score = 0.0
    checks = 0

    if "should_contain" in expected:
        for keyword in expected["should_contain"]:
            checks += 1
            if keyword.lower() in response_text.lower():
                score += 1

    if "should_use_tool" in expected:
        checks += 1
        if expected["should_use_tool"].replace("_", " ").lower() in response_text.lower():
            score += 1

    return score / checks if checks > 0 else 1.0


# ── Score production baseline and push to Langfuse 1-by-1 ────────────

print("=" * 70)
print("KEYWORD ACCURACY: scoring production baseline")
print("=" * 70)

prod_keyword_scores: list[float] = []

for i, r in enumerate(production_results):
    kw_score = score_keyword_accuracy(r["response"], r["expected"])
    prod_keyword_scores.append(kw_score)
    status = "PASS" if kw_score >= 0.5 else "FAIL"
    print(f"  [{status}] {kw_score:.2f}  {r['query'][:55]}")

    # Push score to the exact trace from the dataset run
    if i < len(prod_trace_ids):
        langfuse.create_score(
            trace_id=prod_trace_ids[i],
            name="keyword_accuracy",
            value=kw_score,
            comment=f"Keyword match: {kw_score:.2f}",
        )

langfuse.flush()

avg_kw = sum(prod_keyword_scores) / len(prod_keyword_scores)
pass_rate = sum(1 for s in prod_keyword_scores if s >= 0.5) / len(prod_keyword_scores)
print(f"\nAvg keyword score: {avg_kw:.3f} | Pass rate (>=0.5): {pass_rate:.1%}")
print(f"Pushed {min(len(prod_trace_ids), len(prod_keyword_scores))} scores to Langfuse.")

## Demo 4: LLM-as-Judge Scoring

Keyword matching is brittle -- the agent might provide a correct answer using different wording. An **LLM-as-Judge** evaluator uses a cheaper model (Haiku) to assess response quality on a 0-10 scale.

<div class="alert alert-block alert-info">
<b>Cost tip:</b> We use <code>Claude Haiku 4.5</code> as the judge model for this workshop to keep evaluation costs under $0.01 per run.
</div>

<div class="alert alert-block alert-warning">
<b>Production recommendation:</b> For production evaluations, use a <b>stronger model as the judge</b> (e.g., Claude Opus or Sonnet) to get more accurate and nuanced quality assessments. The judge model should ideally be more capable than the model being evaluated. Haiku is suitable for workshop demos and rapid iteration, but production quality gates benefit from a more discerning judge.
</div>

In [ ]:
# ── LLM-as-Judge evaluator ────────────────────────────────────────────

import re


def evaluate_helpfulness(query: str, response_text: str) -> dict:
    """Use Claude Haiku to score helpfulness on a 0-10 scale.

    Returns a dict with 'score' (normalized 0-1) and 'reason'.
    """
    eval_prompt = f"""Rate the helpfulness of this customer support response on a scale of 0 to 10.

Customer Query: {query}

Agent Response: {response_text}

Scoring criteria:
- 0-3: Unhelpful, incorrect, or confusing
- 4-6: Partially helpful, missing key information
- 7-9: Helpful, accurate, addresses the query well
- 10: Exceptional, exceeds expectations

Return ONLY a JSON object with this exact format:
{{"score": <number>, "reason": "<one sentence explanation>"}}"""

    try:
        resp = call_bedrock(
            prompt=eval_prompt,
            model_id=JUDGE_MODEL_ID,
            max_tokens=300,
            temperature=0.0,
        )
        raw = resp["text"].strip()

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            json_match = re.search(r'\{[^{}]*"score"\s*:\s*\d+[^{}]*\}', raw)
            if json_match:
                parsed = json.loads(json_match.group(0))
            else:
                return {"score": 0.0, "reason": f"No JSON found in: {raw[:80]}"}

        return {
            "score": float(parsed.get("score", 0)) / 10.0,
            "reason": parsed.get("reason", ""),
        }
    except (json.JSONDecodeError, KeyError, ValueError):
        return {"score": 0.0, "reason": f"Parse error. Raw: {resp['text'][:80]}"}


# ── Score production baseline with LLM judge and push to Langfuse 1-by-1 ─

print("=" * 70)
print("LLM-AS-JUDGE: scoring production baseline")
print("=" * 70)

prod_judge_scores: list[dict] = []

for i, r in enumerate(production_results):
    judge_result = evaluate_helpfulness(r["query"], r["response"])
    prod_judge_scores.append(judge_result)
    print(
        f"  {judge_result['score']:.2f}  {r['query'][:45]:<45}  "
        f"{judge_result['reason'][:50]}"
    )

    # Push score to the exact trace from the dataset run
    if i < len(prod_trace_ids):
        langfuse.create_score(
            trace_id=prod_trace_ids[i],
            name="helpfulness_llm",
            value=judge_result["score"],
            comment=judge_result["reason"],
        )

langfuse.flush()

avg_judge = sum(s["score"] for s in prod_judge_scores) / len(prod_judge_scores)
print(f"\nAvg judge score: {avg_judge:.3f} | Items >= 0.7: {sum(1 for s in prod_judge_scores if s['score'] >= 0.7)}/{len(prod_judge_scores)}")
print(f"Pushed {min(len(prod_trace_ids), len(prod_judge_scores))} scores to Langfuse.")

## Demo 5: Run Draft Candidate with Automated Scoring + Quality Gate

Now that you understand how scoring works, let's run the **draft candidate** with evaluators built into `dataset.run_experiment()`. This is the production-ready pattern: the Langfuse SDK automatically creates traces, links them to dataset items, and runs your evaluators -- all in one call.

After the experiment, we compare draft vs production and run the quality gate.

In [ ]:
# ── Run draft candidate with built-in evaluators ─────────────────────

from langfuse import Evaluation

draft_results: list[dict] = []
judge_cache: dict[str, dict] = {}


def draft_task(*, item, **kwargs):
    """Task function: call Bedrock with the draft prompt."""
    query = item.input["query"]
    resp = call_bedrock(
        prompt=query,
        system=system_prompt_draft,
        max_tokens=config_draft.get("max_tokens", 1000),
    )

    # Report model usage to Langfuse for cost tracking
    with langfuse.start_as_current_observation(
        as_type="generation",
        name="bedrock-converse",
        model=MODEL_ID,
        input=query,
    ) as gen:
        gen.update(
            output=resp["text"],
            usage={"input": resp["input_tokens"], "output": resp["output_tokens"]},
        )

    # Pre-compute scores for local results
    kw_score = score_keyword_accuracy(resp["text"], item.expected_output or {})
    judge_result = evaluate_helpfulness(query, resp["text"])
    judge_cache[query] = judge_result

    draft_results.append({
        "query": query,
        "response": resp["text"],
        "input_tokens": resp["input_tokens"],
        "output_tokens": resp["output_tokens"],
        "latency_ms": resp["latency_ms"],
        "expected": item.expected_output or {},
        "keyword_score": kw_score,
        "judge_score": judge_result["score"],
        "judge_reason": judge_result.get("reason", ""),
    })
    return resp["text"]


def keyword_evaluator(*, output, expected_output, **kwargs):
    return Evaluation(
        name="keyword_accuracy",
        value=score_keyword_accuracy(output, expected_output or {}),
    )


def judge_evaluator(*, input, output, **kwargs):
    query = input.get("query", "")
    cached = judge_cache.get(query)
    if cached:
        return Evaluation(
            name="helpfulness_llm",
            value=cached["score"],
            comment=cached.get("reason", ""),
        )
    result = evaluate_helpfulness(query, output)
    return Evaluation(
        name="helpfulness_llm",
        value=result["score"],
        comment=result.get("reason", ""),
    )


print("Running experiment: draft-candidate (with automated scoring)...")
langfuse_dataset.run_experiment(
    name="draft-candidate",
    task=draft_task,
    evaluators=[keyword_evaluator, judge_evaluator],
    max_concurrency=1,
)
langfuse.flush()
print(f"  Completed {len(draft_results)} items (scores auto-pushed to Langfuse)")

# ── Compare draft vs production ───────────────────────────────────────

draft_kw = [r["keyword_score"] for r in draft_results]
draft_jd = [{"score": r["judge_score"], "reason": r["judge_reason"]} for r in draft_results]


def compute_stats(kw_scores, jd_scores):
    combined = [(kw + jd["score"]) / 2.0 for kw, jd in zip(kw_scores, jd_scores)]
    return {
        "avg_keyword": sum(kw_scores) / len(kw_scores),
        "avg_judge": sum(s["score"] for s in jd_scores) / len(jd_scores),
        "avg_combined": sum(combined) / len(combined),
        "pass_rate": f"{sum(1 for s in combined if s >= 0.5)}/{len(combined)}",
    }


stats_draft = compute_stats(draft_kw, draft_jd)
stats_prod = compute_stats(prod_keyword_scores, prod_judge_scores)

print(f"\n{'=' * 70}")
print("COMBINED SCORE COMPARISON: draft vs production")
print(f"{'=' * 70}")
print(f"\n{'Metric':<28} {'draft':>14} {'production':>14}")
print("-" * 60)
print(f"{'Avg keyword score':<28} {stats_draft['avg_keyword']:>14.3f} {stats_prod['avg_keyword']:>14.3f}")
print(f"{'Avg judge score':<28} {stats_draft['avg_judge']:>14.3f} {stats_prod['avg_judge']:>14.3f}")
print(f"{'Avg combined score':<28} {stats_draft['avg_combined']:>14.3f} {stats_prod['avg_combined']:>14.3f}")
print(f"{'Pass rate (combined >= 0.5)':<28} {stats_draft['pass_rate']:>14} {stats_prod['pass_rate']:>14}")

# Token savings
avg_in_draft = sum(r["input_tokens"] for r in draft_results) / len(draft_results)
avg_in_prod = sum(r["input_tokens"] for r in production_results) / len(production_results)
token_savings = (1 - avg_in_draft / avg_in_prod) * 100 if avg_in_prod > 0 else 0
print(f"{'Avg input tokens':<28} {avg_in_draft:>12.0f}tk {avg_in_prod:>12.0f}tk")
print(f"{'Input token savings':<28} {token_savings:>13.1f}%")
print("=" * 70)

# ── Quality gate ──────────────────────────────────────────────────────
# Draft must meet absolute threshold AND not regress from production.

SCORE_THRESHOLD = 0.7
avg_draft = stats_draft["avg_combined"]
avg_prod = stats_prod["avg_combined"]

if avg_draft < SCORE_THRESHOLD:
    print(f"\nQuality gate: FAILED (draft avg {avg_draft:.3f} < threshold {SCORE_THRESHOLD})")
    print("Action: Draft does not meet minimum quality. Iterate on the prompt.")
elif avg_draft < avg_prod:
    print(f"\nQuality gate: FAILED (draft avg {avg_draft:.3f} < production avg {avg_prod:.3f})")
    print("Action: Draft regresses quality compared to production. Do not promote.")
else:
    print(f"\nQuality gate: PASSED (draft avg {avg_draft:.3f} >= production avg {avg_prod:.3f})")
    print("Draft quality matches or exceeds production -- safe to promote.")

---

# Section 3: CI/CD Integration with AWS CodePipeline (20 min)

In production, prompt changes should go through the same rigor as code changes: automated testing, quality gates, and approval workflows. In this section you will:

1. **Review** the pipeline components (evaluation script, buildspec, Lambda, CloudFormation)
2. **Configure** your Langfuse credentials for the pipeline
3. **Push** evaluation code to CodeCommit to trigger the pipeline
4. **Monitor** the pipeline execution and review results

The pipeline infrastructure has been pre-deployed by your instructor via CloudFormation. Your job is to wire up credentials and trigger it.

| Concept | Description |
|---------|-------------|
| **Langfuse as quality gate** | CI/CD checks Langfuse scores before deployment |
| **External evaluation pipeline** | Script that fetches dataset, runs evals, pushes scores |
| **CodePipeline + Lambda** | AWS-native CI/CD with serverless evaluation |

**References:**
- [AWS CodePipeline User Guide](https://docs.aws.amazon.com/codepipeline/latest/userguide/welcome.html)
- [Langfuse External Evaluation Pipelines](https://langfuse.com/docs/scores/external-evaluation-pipelines)

## Pipeline Architecture

The instructor has pre-deployed a CodePipeline via CloudFormation. Here is what it contains:

```
+---------------------------------------------------------------------------+
|                        AWS CodePipeline                                   |
+---------------------------------------------------------------------------+
|                                                                           |
|  +----------+    +-----------+    +-----------+    +----------+            |
|  |  Source  |--->|   Build   |--->| Quality   |--->| Approval |--->Deploy  |
|  |(CodeCommit)|  |(CodeBuild)|   |   Gate    |    | (Manual) |            |
|  +----------+    +-----------+    +-----------+    +----------+            |
|       |                |                                                  |
|       |                v                                                  |
|       |       +-----------------+                                         |
|       |       |    Langfuse     |                                         |
|       |       |  (Self-Hosted)  |                                         |
|       |       |                 |                                         |
|       |       | - Fetch dataset |                                         |
|       |       | - Run evals     |                                         |
|       |       | - Push scores   |                                         |
|       |       +-----------------+                                         |
|       |                                                                   |
|  Scripts pushed by you:                                                   |
|  - scripts/evaluate_prompt.py  (agent eval + LLM judge + score push)      |
|  - scripts/check_quality_gate.py  (threshold check, exit 0 or 1)         |
+---------------------------------------------------------------------------+
```

**Pre-deployed resources:**

| Resource | Type | Purpose |
|----------|------|---------|
| `prompt-repo` | CodeCommit | You push evaluation scripts here |
| `langfuse-pipeline-credentials` | Secrets Manager | You populate with your Langfuse keys |
| `prompt-evaluation` | CodeBuild | Runs `evaluate_prompt.py` + quality gate |
| `prompt-evaluator` | Lambda | CodePipeline deploy action |
| `prompt-evaluation-pipeline` | CodePipeline | Source → Build → Approval → Deploy |

**How it works:**
1. You push scripts to CodeCommit
2. CodeBuild fetches the **evaluation dataset from Langfuse** (not from git)
3. Runs each item through the agent, scores with keyword + LLM judge
4. Quality gate checks thresholds — pipeline fails if they are not met
5. Manual approval before production deployment

## Hands-On: Deploy and Run the Pipeline

Now that you understand the components, let's wire up the pipeline and trigger it.

**Pre-deployed by instructor:**
- CodePipeline (`prompt-evaluation-pipeline`)
- CodeBuild project (`prompt-evaluation`)
- CodeCommit repository (`prompt-repo`)
- Secrets Manager secret (`langfuse-pipeline-credentials`) — placeholder values

**Your steps:**
1. Populate the Secrets Manager secret with your Langfuse API keys
2. Push evaluation scripts and dataset to CodeCommit
3. Trigger the pipeline and monitor execution

### Step 1: Populate Langfuse Credentials in Secrets Manager

The pipeline needs your Langfuse API keys to fetch datasets and push scores. Your keys are already loaded from `.env` — this cell stores them in Secrets Manager where CodeBuild can access them.

In [ ]:
# ── Store Langfuse credentials in Secrets Manager ─────────────────────

sm = boto3.client("secretsmanager", region_name=REGION)

sm.put_secret_value(
    SecretId="langfuse-pipeline-credentials",
    SecretString=json.dumps({
        "public_key": os.environ["LANGFUSE_PUBLIC_KEY"],
        "secret_key": os.environ["LANGFUSE_SECRET_KEY"],
        "host": os.environ.get("LANGFUSE_BASE_URL", ""),
    }),
)
print("Langfuse credentials stored in Secrets Manager: langfuse-pipeline-credentials")
print(f"  Host: {os.environ.get('LANGFUSE_BASE_URL', 'N/A')}")

### Step 2: Push Evaluation Scripts to CodeCommit

The pipeline's CodeBuild stage needs `scripts/evaluate_prompt.py` and `scripts/check_quality_gate.py` in the repository. The **evaluation dataset lives in Langfuse** (uploaded in Section 2) — the pipeline fetches it at runtime, so there's no need to store it in git.

In [ ]:
# ── Push evaluation scripts to CodeCommit ─────────────────────────────
# Note: The evaluation dataset is already in Langfuse (uploaded in Section 2).
# The pipeline fetches it directly from Langfuse — no need to push the JSON file.

REPO_NAME = "prompt-repo"

codecommit = boto3.client("codecommit", region_name=REGION)

# Only push the scripts — dataset lives in Langfuse
files_to_push = [
    ("scripts/evaluate_prompt.py", "scripts/evaluate_prompt.py"),
    ("scripts/check_quality_gate.py", "scripts/check_quality_gate.py"),
]

put_files = []
for local_path, repo_path in files_to_push:
    content = Path(local_path).read_bytes()
    put_files.append({"filePath": repo_path, "fileContent": content})

# Push to CodeCommit (handles both initial and subsequent commits)
try:
    branch = codecommit.get_branch(repositoryName=REPO_NAME, branchName="main")
    parent_commit = branch["branch"]["commitId"]
    response = codecommit.create_commit(
        repositoryName=REPO_NAME,
        branchName="main",
        parentCommitId=parent_commit,
        putFiles=put_files,
        commitMessage="Update evaluation scripts",
    )
except codecommit.exceptions.BranchDoesNotExistException:
    response = codecommit.create_commit(
        repositoryName=REPO_NAME,
        branchName="main",
        putFiles=put_files,
        commitMessage="Initial commit: evaluation scripts",
    )

print(f"Pushed {len(put_files)} files to CodeCommit: {REPO_NAME}")
print(f"  Commit: {response['commitId'][:12]}...")
print(f"\n  Dataset '{DATASET_NAME}' is fetched from Langfuse at runtime — not stored in git.")

### Step 3: Trigger and Monitor the Pipeline

The CodeCommit push above may trigger the pipeline automatically (via EventBridge). If not, we start it manually and poll for status.

In [ ]:
# ── Trigger pipeline and monitor status ───────────────────────────────

PIPELINE_NAME = "prompt-evaluation-pipeline"

cp = boto3.client("codepipeline", region_name=REGION)

# Start pipeline execution
execution = cp.start_pipeline_execution(name=PIPELINE_NAME)
exec_id = execution["pipelineExecutionId"]
print(f"Pipeline triggered: {exec_id}")
print(f"Console: https://{REGION}.console.aws.amazon.com/codesuite/codepipeline/pipelines/{PIPELINE_NAME}/view")

# Poll for status (check every 30 seconds, up to 10 minutes)
print("\nMonitoring pipeline execution...")
for attempt in range(20):
    time.sleep(30)
    state = cp.get_pipeline_execution(
        pipelineName=PIPELINE_NAME,
        pipelineExecutionId=exec_id,
    )
    status = state["pipelineExecution"]["status"]
    print(f"  [{attempt * 30:>3}s] Status: {status}")

    if status in ("Succeeded", "Failed", "Stopped", "Cancelled"):
        break

# Show final stage details
pipeline_state = cp.get_pipeline_state(name=PIPELINE_NAME)
print(f"\n{'Stage':<25} {'Status':<15} {'Last Updated'}")
print("-" * 65)
for stage in pipeline_state["stageStates"]:
    stage_name = stage["stageName"]
    action = stage.get("actionStates", [{}])[0]
    action_status = action.get("latestExecution", {}).get("status", "N/A")
    last_updated = action.get("latestExecution", {}).get("lastStatusChange", "")
    ts = str(last_updated)[:19] if last_updated else "N/A"
    print(f"  {stage_name:<23} {action_status:<15} {ts}")

if status == "Succeeded":
    print("\nPipeline PASSED -- prompt evaluation met quality gates.")
    print("In production, the Approval stage would now wait for a human reviewer.")
elif status == "Failed":
    print("\nPipeline FAILED -- check CodeBuild logs for evaluation details.")
else:
    print(f"\nPipeline status: {status} -- check the console for details.")

## Beyond CodePipeline

The evaluation scripts (`evaluate_prompt.py` and `check_quality_gate.py`) are **platform-agnostic** — they only need Python, boto3, and Langfuse credentials. You can run them in any CI/CD platform:

- **GitHub Actions** — trigger on PR changes to `prompts/` or `agents/` directories
- **GitLab CI/CD** — add as a pipeline stage before deployment
- **Jenkins** — run as a build step with credentials from a secrets vault
- **Bitbucket Pipelines** — similar to GitHub Actions with environment variables

The pattern is the same everywhere: fetch dataset from Langfuse → run evaluation → check quality gate → deploy or reject.

---

## Summary

### Section 1: Prompt Management

| Concept | What You Learned |
|---------|------------------|
| Prompt versioning | Store versions in Langfuse with `draft` / `production` labels |
| Label-based routing | Fetch prompts by label at runtime — decouple code deploys from prompt changes |
| Prompt-trace linkage | Every trace records which prompt version produced it |

### Section 2: Evaluation & Quality Scoring

| Concept | What You Learned |
|---------|------------------|
| Datasets | Store test cases in Langfuse as the single source of truth |
| `run_experiment()` | Run all dataset items through an agent, auto-link traces to items |
| Keyword scoring | Fast, deterministic check for expected content — push scores manually |
| LLM-as-Judge | Use a cheaper model (Haiku) for nuanced 0-10 quality assessment |
| Evaluators | Embed scoring functions in `run_experiment()` for automated scoring |
| Quality gate | Draft must meet absolute threshold AND not regress from production |

### Section 3: CI/CD Integration

| Concept | What You Learned |
|---------|------------------|
| Dataset as source of truth | Pipeline fetches dataset from Langfuse at runtime — not stored in git |
| Secrets Manager | Store Langfuse credentials for CodeBuild access |
| CodePipeline | Source → Build (eval + gate) → Approval → Deploy |
| Platform-agnostic scripts | `evaluate_prompt.py` and `check_quality_gate.py` run anywhere |

<div class="alert alert-block alert-info">
<b>Key takeaway:</b> Treat prompt changes with the same rigor as code changes — version them, evaluate systematically, gate on quality, and automate the pipeline.
</div>

---

## Additional Resources

- [Langfuse Prompt Management](https://langfuse.com/docs/prompts/get-started)
- [Langfuse Datasets & Experiments](https://langfuse.com/docs/datasets/overview)
- [Langfuse Scores & Model-Based Evals](https://langfuse.com/docs/scores/model-based-evals)
- [Langfuse External Evaluation Pipelines](https://langfuse.com/docs/scores/external-evaluation-pipelines)
- [AWS CodePipeline User Guide](https://docs.aws.amazon.com/codepipeline/latest/userguide/welcome.html)
- [CodeBuild Buildspec Reference](https://docs.aws.amazon.com/codebuild/latest/userguide/build-spec-ref.html)